TASK 1 — Data Loading & Exploration

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('Housing.csv')

# Display the first 10 rows
df.head(10)

TASK 2 — Data Cleaning

In [ ]:
# Check the shape of the dataset (rows, columns)
print("Dataset Shape:", df.shape)

# List all columns to identify features and target
print("\nColumns in the dataset:")
print(df.columns)

In [ ]:
# Check for missing values in each column
print("Missing values in each column:")
print(df.isnull().sum())

In [ ]:
# Step 1: Check original shape and drop duplicate rows
print("Original dataset shape:", df.shape)
df = df.drop_duplicates()
print("Shape after dropping duplicates:", df.shape)

# Step 2: Identify categorical columns (columns with text)
categorical_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 
                    'airconditioning', 'prefarea', 'furnishingstatus']

# Step 3: Apply One-Hot Encoding
# drop_first=True use chesthe redundant columns create avvavu (e.g., yes/no ki okkati chalu)
# dtype=int use chesthe True/False badulu 1/0 vastayi
df_cleaned = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)

# Step 4: Display the new cleaned dataset
print("\nNew shape after One-Hot Encoding:", df_cleaned.shape)
display(df_cleaned.head())

TASK 3 — Model Building

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Step 1: Split data into Features (X) and Target (y)
X = df_cleaned.drop('price', axis=1) # Price thappa migilina anni features
y = df_cleaned['price'] # Target column is price

# Step 2: Split the data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: Train and Evaluate Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_predictions = lr_model.predict(X_test)

print("--- Linear Regression Performance ---")
print(f"MAE: {mean_absolute_error(y_test, lr_predictions):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, lr_predictions)):.2f}")
print(f"R² Score: {r2_score(y_test, lr_predictions):.4f}\n")

# Step 4: Train and Evaluate Random Forest Model
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

print("--- Random Forest Performance ---")
print(f"MAE: {mean_absolute_error(y_test, rf_predictions):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, rf_predictions)):.2f}")
print(f"R² Score: {r2_score(y_test, rf_predictions):.4f}")

TASK 4 — Visualization (Histogram & Correlation heatmap)

In [ ]:
# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# Step 1: Load a fresh dataset to prevent any missing column errors
df = pd.read_csv('Housing.csv')

# Step 2: Create a numeric version of the data for the Heatmap
# Heatmaps require numbers, so we quickly encode the text columns (yes/no) into 1s and 0s
categorical_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 
                    'airconditioning', 'prefarea', 'furnishingstatus']
df_numeric = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)

# Step 3: Define a custom function to format large numbers (e.g., 10,000,000 instead of 1e7)
def format_numbers(x, pos):
    return f'{int(x):,}'

# Set the overall visual style
sns.set_theme(style="whitegrid")

# Create a figure with 2 vertical subplots
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(10, 14))

# --- Chart 1: Histogram showing the distribution of house prices ---
sns.histplot(df['price'], kde=True, color='royalblue', ax=axes[0])
axes[0].set_title('Distribution of House Prices', fontsize=14, fontweight='bold')
axes[0].set_xlabel('House Price')
axes[0].set_ylabel('Frequency')
# Apply the number formatter to remove scientific notation
axes[0].xaxis.set_major_formatter(FuncFormatter(format_numbers))

# --- Chart 2: Correlation Heatmap ---
# Calculate the correlation matrix using the strictly numeric dataframe
correlation_matrix = df_numeric.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", ax=axes[1])
axes[1].set_title('Correlation Heatmap of Housing Features', fontsize=14, fontweight='bold')

# Adjust layout spacing to keep everything clean and readable
plt.tight_layout()

# Display the charts
plt.show()

TASK 4 — Visualization (Price Variance by Categorical Features)

In [ ]:
# Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import warnings
warnings.filterwarnings('ignore')

# Step 1: Load a fresh dataset to avoid any missing data errors
df = pd.read_csv('Housing.csv')

# Step 2: Define a function to remove scientific notation (1e7) from the Y-axis
def format_numbers(x, pos):
    return f'{int(x):,}'

# Step 3: List all the categorical features we want to compare with Price
categorical_features = [
    'bedrooms', 'bathrooms', 'stories', 'mainroad', 
    'guestroom', 'basement', 'hotwaterheating', 
    'airconditioning', 'parking', 'prefarea', 'furnishingstatus'
]

# Set the visual style
sns.set_theme(style="whitegrid")

# Step 4: Create a large grid layout (4 rows, 3 columns = 12 total spaces)
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(15, 20))
axes = axes.flatten() # Flatten to easily loop through them

# Step 5: Loop through each feature and draw a boxplot
for i, col in enumerate(categorical_features):
    sns.boxplot(x=df[col], y=df['price'], ax=axes[i], palette='Set2')
    
    # Add clear titles and labels
    axes[i].set_title(f'Price vs {col.capitalize()}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col.capitalize())
    axes[i].set_ylabel('House Price')
    
    # Apply the number formatting to the Y-axis
    axes[i].yaxis.set_major_formatter(FuncFormatter(format_numbers))

# Step 6: Since we have 11 features but 12 spaces, safely remove the last empty box
for j in range(len(categorical_features), len(axes)):
    fig.delaxes(axes[j])

# Adjust spacing so titles and labels don't overlap
plt.tight_layout()

# Show the final beautiful dashboard!
plt.show()

**Task 5: Insights & Summary**

Based on the data analysis, the features that most strongly influence a house's price are the total area, number of bathrooms, presence of air conditioning, and the number of stories. Regarding accuracy, both the Linear Regression and Random Forest models performed similarly with around 65% accuracy (R² score), meaning they are moderately accurate but still have an average prediction error of roughly 9.5 Lakhs. What surprised me in the data was that advanced models like Random Forest didn't significantly outperform simple Linear Regression, and features we usually assume add high value (like basements or guestrooms) have a surprisingly weak correlation with the final price. Recommendation: A real estate business should strongly highlight properties with multiple bathrooms and air conditioning in their marketing campaigns, as the data clearly shows buyers are willing to pay a substantial premium for these specific amenities.